# 03. Supply and demand

Where builders kept pace, prices rose less. The regression that says so has one objection built
into it: builders build where people are arriving, and a regression that cannot see the arrivals
credits the building with what the demand did. This notebook keeps the dead end: the specification
that put the demand shock in the wrong fixed effect. It averaged each metro's migration over the
whole window set and entered that as the control, which the metro fixed effect absorbs entirely, so
the controlled and the uncontrolled estimates came out identical and the control did nothing.

First on the simulator, where the elasticity is planted; then the real cells.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(3)
ROOT = Path(os.environ.get("DOORSTEP_NOTEBOOK_ROOT") or (Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent))
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values, tables = manifest["values"], manifest["tables"]


def v(key):
    return values[key]["value"]


def t(key):
    # A manifest value formatted the way the documents print it.
    from doorstep_core.formats import format_value

    return format_value(values[key]["value"], values[key]["fmt"])


def table(key):
    entry = tables[key]
    return pl.DataFrame(entry["rows"], schema=entry["columns"], orient="row")


MARTS = ROOT / "results" / "marts"
print("manifest as of", manifest["as_of"], "with", len(values), "values; prices to", v("data.fhfa.latest_period"), "and the census to", v("data.acs.last_year"))

manifest as of 2026-10-22 with 923 values; prices to 2026Q2 and the census to 2024


## On the simulator

Three specifications on the same cells: no demand control, the dead end (a metro level mean of
migration, absorbed by the metro effect), and the kept one (migration over the window and the two
years before it, which varies within the metro).

In [2]:
from doorstep_chapters import c4_supply
from doorstep_chapters.panel import from_world
from doorstep_sim.metros import BETA_SUPPLY, Condition, generate
from doorstep_stats.regression import first, regress

strong = Condition(name="strong", demand_strength="strong", supply_dispersion="narrow", reverse_causality="none", noise="low")
rows = []
for seed in range(4):
    world = generate(strong, seed)
    cells = c4_supply.cells(from_world(world), 5)
    y = cells["annual_growth"].to_numpy()
    w = cells["population"].to_numpy()
    fe = [cells["cbsa_code"].to_numpy(), cells["window_id"].to_numpy()]
    cl = cells["state"].to_numpy()
    metro_mean = cells.join(cells.group_by("cbsa_code").agg(pl.col("migration_rate").mean().alias("metro_mean")), on="cbsa_code")["metro_mean"].to_numpy()
    none = first(regress(y, cells["permits_rate"].to_numpy()[:, None], ["permits_rate"], weights=w, fixed_effects=fe, cluster=cl), "permits_rate")
    try:
        dead_end = first(regress(y, np.column_stack([cells["permits_rate"].to_numpy(), metro_mean]), ["permits_rate", "metro_mean"], weights=w, fixed_effects=fe, cluster=cl), "permits_rate")
        dead_end_estimate = dead_end.estimate
    except Exception as err:  # the solver refuses a control the fixed effects absorb
        dead_end_estimate = float("nan")
        if seed == 0:
            print("dead end:", err)
    kept, _ = c4_supply.fit_both(cells)
    rows.append({"seed": seed, "no control": none.estimate, "dead end": dead_end_estimate, "kept": kept.estimate})
frame = pl.DataFrame(rows)
print("planted elasticity", -BETA_SUPPLY)
frame.select(pl.all().exclude("seed").mean())

dead end: a regressor is collinear with the fixed effects


planted elasticity -0.025


no control,dead end,kept
f64,f64,f64
-0.022,NaN,-0.025


The dead end does not even run: a control that does not vary within the metro is collinear with the
metro effect, and the solver refuses it rather than returning the uncontrolled estimate under a
different name, which is what the first version of this chapter had silently done. The kept
specification moves toward the planted value under strong demand; the recovery study on the data
page reports the remaining attenuation by condition.

## The real cells

Both specifications from the manifest, the window interior test, and the cells themselves.

In [3]:
table("ch4.specifications")

Specification,Annual growth per permit per 100 households,Low,High,Over the window,Clusters
str,f64,f64,f64,f64,i64
"""with the demand control""",-0.021,-0.028,-0.015,-0.107,9
"""without the demand control""",-0.017,-0.024,-0.009,-0.083,9


In [4]:
table("ch4.window_interior")

Window years,Controlled,Low,High,Uncontrolled,Note
i64,f64,f64,f64,f64,str
3,-0.023,-0.025,-0.021,-0.020,""""""
5,-0.021,-0.028,-0.015,-0.017,"""chosen"""
7,-0.020,-0.022,-0.018,-0.017,""""""


In [5]:
cells = pl.read_parquet(MARTS / "ch4_cells.parquet")
print(cells.height, "cells over", cells["cbsa_code"].n_unique(), "metros and", cells["window_id"].n_unique(), "windows")
cells.group_by("window_id").agg(
    pl.col("first_year").min(), pl.col("last_year").max(), pl.col("permits_rate").mean().alias("mean building rate"),
    pl.col("annual_growth").mean().alias("mean annual growth"), pl.col("migration_rate").mean().alias("mean migration per 100"),
).sort("window_id")

180 cells over 60 metros and 3 windows


window_id,first_year,last_year,mean building rate,mean annual growth,mean migration per 100
i64,i64,i64,f64,f64,f64
1,2006,2010,1.164,0.009,0.285
2,2011,2015,1.252,0.030,0.150
3,2016,2020,1.267,0.035,-0.074
